# Exercise 5.3 — Decision trees, overfitting and tuning

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day5/day5-03-decision-trees.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

A decision tree predicts by asking yes-or-no questions: is glucose above 128?
If so, is body mass above 30? Each answer sends a person down one branch until
they reach a **leaf**, which gives a predicted class and a probability. Trees
are easy to read. Their weakness is that a tree can keep splitting until it has
memorised the training rows. In this exercise we draw a small tree, watch a
large one overfit, and then let a grid search choose how large the tree may
grow.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas matplotlib scikit-learn

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

The 750 people in [`data/diabetes_clean.csv`](data/diabetes_clean.csv), with
five predictors: pregnant, glucose, mass, pedigree and age. The outcome is 1
for `pos` and 0 for `neg`. We keep 80% of the people for training (600) and
hold back 20% as the test set (150). `random_state=42` makes the split the same
every time.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

train, test = train_test_split(diabetes, test_size=0.2, random_state=42)

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

------------------------------------------------------------------------

### Part 1 — A small tree

Fit a tree that may ask at most three questions in a row (`max_depth=3`).
`random_state` fixes how ties between equally good splits are broken.

In [ ]:
tree_small = DecisionTreeClassifier(max_depth=3, random_state=123)
tree_small = tree_small.fit(train[predictors], train["diabetes"])

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
drawn = plot_tree(tree_small, feature_names=predictors, class_names=["neg", "pos"],
                  filled=True, proportion=True, impurity=False, fontsize=9, ax=ax)
plt.show()

Each box shows the question, the share of training rows that reach it
(`samples`), the share of `neg` and `pos` among them (`value`) and the
predicted class. The first question is about glucose. Now take the first person
in the test set:

In [ ]:
patient = test.iloc[[0]]
patient

Follow this person down the tree by hand: at each box, go left if the answer is
true. Which leaf do they reach, and what probability of diabetes does it give?
Then check your answer with `predict_proba()`.

In [ ]:
# Replace every ______ below, then run this cell.
tree_small.predict_proba(______[predictors])

> **Stuck?**
>
> `patient` holds one row; pass its predictor columns.

------------------------------------------------------------------------

### Part 2 — A tree that nothing stops overfits

With its default settings, `DecisionTreeClassifier()` keeps splitting until
every leaf holds only one class. Nothing stops the tree.

In [ ]:
tree_full = DecisionTreeClassifier(random_state=123)
tree_full = tree_full.fit(train[predictors], train["diabetes"])

print("leaves:", tree_full.get_n_leaves())

135 leaves for 600 people: far too many to draw. We judge it with two numbers:
accuracy, and the **AUC**. The AUC is the chance that a random person with
diabetes gets a higher predicted probability than a random person without: 0.5
is guessing, 1 is perfect. Score the tree on the training rows and on the
test rows:

In [ ]:
# Replace every ______ below, then run this cell.
for name, data in [("train", train), ("test", ______)]:
    prob  = tree_full.predict_proba(data[predictors])[:, 1]   # probability of pos
    label = tree_full.predict(data[predictors])               # class, threshold 0.5
    print(name,
          "accuracy", round(accuracy_score(data["diabetes"], label), 3),
          "AUC", round(roc_auc_score(data["diabetes"], prob), 3))

> **Stuck?**
>
> Only the data changes: the test rows are in `test`.

**Question!** What has the tree learnt from the training rows that does not
carry over to new people?

------------------------------------------------------------------------

### Part 3 — Choosing the tree size with a grid search

How far a tree may grow is set by **hyperparameters**, like the penalty in
Exercise 5.2:

- `max_depth`: the most questions from the top of the tree to a leaf;
- `min_samples_split`: the fewest people a node must hold before it may be
  split.

In Exercise 5.2 each candidate was scored on one validation set. Here we use
**5-fold cross-validation**: the training rows are cut into 5 parts (folds).
Each combination is fitted on four folds and scored on the fifth, five times
over, so that every training row is used once for scoring. The five scores are
averaged. `cv=5` makes the folds and keeps the share of people with diabetes
the same in every fold; `scoring="roc_auc"` scores each combination by its AUC. The grid has five
depths times five node sizes, 25 combinations.

In [ ]:
tree_grid = {"max_depth":         [1, 3, 5, 7, 10],
             "min_samples_split": [2, 9, 16, 23, 30]}

tree_search = GridSearchCV(DecisionTreeClassifier(random_state=123),
                           param_grid=tree_grid, cv=5, scoring="roc_auc")
tree_search = tree_search.fit(train[predictors], train["diabetes"])

print(tree_search.best_params_)
print("cross-validated AUC:", round(tree_search.best_score_, 3))

In [ ]:
tree_results = pd.DataFrame(tree_search.cv_results_)
fig, ax = plt.subplots()
for split in tree_grid["min_samples_split"]:
    rows = tree_results[tree_results["param_min_samples_split"] == split]
    ax.plot(rows["param_max_depth"], rows["mean_test_score"], marker="o", label=str(split))
ax.set_xlabel("max_depth")
ax.set_ylabel("Mean cross-validated AUC")
ax.legend(title="min_samples_split")
plt.show()

A single question (depth 1) is too simple. When small nodes may be split,
deep trees overfit and the AUC falls. `GridSearchCV()` has already refitted the
best combination on all 600 training rows:

In [ ]:
final_tree = tree_search.best_estimator_

Now, and only now, use the test rows. Score the chosen tree once, next to its
score on the training rows.

In [ ]:
# Replace every ______ below, then run this cell.
for name, data in [("train", train), ("test", ______)]:
    prob  = final_tree.predict_proba(data[predictors])[:, 1]
    label = final_tree.predict(data[predictors])
    print(name,
          "accuracy", round(accuracy_score(data["diabetes"], label), 3),
          "AUC", round(roc_auc_score(data["diabetes"], ______), 3))

> **Stuck?**
>
> The test rows are in `test`; the AUC needs the predicted probabilities, `prob`.

**Question!** Why must the test rows play no part in choosing `max_depth` and
`min_samples_split`?

> **Key points**
>
> - A tree asks one yes-or-no question at a time; each leaf gives a predicted class and a probability.
> - A tree that nothing stops is perfect on its training rows and much worse on the test rows: it overfits.
> - Tree depth and minimum node size are hyperparameters. Choose them with a grid search scored by cross-validation on the training rows.
> - Fit the chosen tree on all the training rows, then check it once on the test rows.